# Checkpoint 8 — refined tiers plus clean channel average

This experiment keeps the refined subscriber tiers, keeps raw subscriber count excluded, and adds historical average channel views per video. The average is present only when the channel snapshot was captured at or before publication; backfilled statistics are missing to prevent target contamination.

The same horizon rows, grouped folds, target transform, model parameters, and untouched reserved test are used for the A/B comparison.

In [1]:
from pathlib import Path
import json
import sys

import joblib
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.train_checkpoint5_models import (
    HORIZONS,
    MODEL_NAME,
    load_horizon_checkpoint,
    train_all_horizons,
)
from viewcastlk_ml.horizon_preprocessing import HorizonDatasetPreprocessor

TIER_ONLY_DIR = PROJECT_ROOT / 'artifacts' / 'checkpoint7_refined_tier_only'
CLEAN_AVERAGE_DIR = PROJECT_ROOT / 'artifacts' / 'checkpoint8_clean_channel_average'
AVERAGE_FEATURE = 'ch_avg_views_per_video_at_publish'

loaded = {}
coverage_rows = []
for horizon in HORIZONS:
    X, y, assignments, _, _ = load_horizon_checkpoint(PROJECT_ROOT, horizon)
    loaded[horizon] = (X, y, assignments)
    values = pd.to_numeric(X[AVERAGE_FEATURE], errors='coerce')
    assert 'ch_views_at_publish' not in X.columns
    assert 'ch_subs_at_publish' not in X.columns
    assert not np.isinf(values.dropna()).any()
    coverage_rows.append({
        'horizon_days': horizon,
        'rows': len(X),
        'clean_average_rows': int(values.notna().sum()),
        'missing_rows': int(values.isna().sum()),
        'coverage_percent': 100 * values.notna().mean(),
        'median_clean_average': values.median(),
    })

coverage = pd.DataFrame(coverage_rows)
display(coverage.round(2))

   horizon_days   rows  ...  coverage_percent  median_clean_average
0             7  20663  ...             66.92              14312.39
1            14  15685  ...             51.47              16537.12
2            21  15100  ...             17.02              16591.77
3            30  14753  ...              0.00                   NaN

[4 rows x 6 columns]


In [2]:
# Visible fold-safe preprocessing example.
X7, y7, assignments7 = loaded[7]
training_mask = assignments7['partition'].eq('development') & ~assignments7['cv_validation_fold'].eq(1)
validation_mask = assignments7['partition'].eq('development') & assignments7['cv_validation_fold'].eq(1)
training_positions = assignments7.loc[training_mask, 'horizon_row_position'].astype(int).to_numpy()
validation_positions = assignments7.loc[validation_mask, 'horizon_row_position'].astype(int).to_numpy()

preprocessor = HorizonDatasetPreprocessor()
preprocessor.fit(X7.iloc[training_positions], np.log1p(y7.iloc[training_positions]))
source_preview = X7.iloc[validation_positions[:10]]
model_preview = preprocessor.transform(source_preview)
tier_features = [column for column in model_preview if column.startswith('subscriber_tier_')]

display(pd.concat([
    source_preview[['subscriber_tier', AVERAGE_FEATURE]].reset_index(drop=True),
    model_preview[[AVERAGE_FEATURE] + tier_features].reset_index(drop=True),
], axis=1))

assert AVERAGE_FEATURE in model_preview.columns
assert 'ch_views_at_publish' not in model_preview.columns
assert 'ch_subs_at_publish' not in model_preview.columns
assert not np.isinf(model_preview.to_numpy(dtype=float)).any()
print('PASS: the clean derived average is retained; both raw channel values remain excluded.')

  subscriber_tier  ...  subscriber_tier_under_1k
0         1m_plus  ...                       0.0
1         1m_plus  ...                       0.0
2         1m_plus  ...                       0.0
3       1k_to_10k  ...                       0.0
4         1m_plus  ...                       0.0
5         1m_plus  ...                       0.0
6      500k_to_1m  ...                       0.0
7         1m_plus  ...                       0.0
8         1m_plus  ...                       0.0
9         1m_plus  ...                       0.0

[10 rows x 10 columns]
PASS: the clean derived average is retained; both raw channel values remain excluded.


In [3]:
training_run = train_all_horizons(
    project_root=PROJECT_ROOT,
    output_dir=CLEAN_AVERAGE_DIR,
    n_estimators=800,
    n_jobs=4,
    include_llm_scores=False,
)

display(training_run['summary'][[
    'horizon_days', 'model', 'rows', 'mape_nonzero_pct',
    'median_ape_nonzero_pct', 'smape_pct', 'rmsle', 'log_r2'
]])


Training independent day-7 model
day 7 fold 1/5: RMSLE=1.9406, median APE=125.12%, best trees=45
day 7 fold 2/5: RMSLE=2.0414, median APE=85.70%, best trees=146
day 7 fold 3/5: RMSLE=2.1065, median APE=93.55%, best trees=48
day 7 fold 4/5: RMSLE=1.9082, median APE=83.38%, best trees=130
day 7 fold 5/5: RMSLE=2.2906, median APE=109.79%, best trees=23

Training independent day-14 model
day 14 fold 1/5: RMSLE=2.0519, median APE=227.93%, best trees=17
day 14 fold 2/5: RMSLE=1.9256, median APE=86.14%, best trees=394
day 14 fold 3/5: RMSLE=2.1987, median APE=96.64%, best trees=143
day 14 fold 4/5: RMSLE=1.9219, median APE=88.52%, best trees=97
day 14 fold 5/5: RMSLE=1.9698, median APE=92.93%, best trees=302

Training independent day-21 model
day 21 fold 1/5: RMSLE=1.9064, median APE=98.64%, best trees=146
day 21 fold 2/5: RMSLE=1.9496, median APE=86.34%, best trees=128
day 21 fold 3/5: RMSLE=2.0570, median APE=90.09%, best trees=48
day 21 fold 4/5: RMSLE=1.9893, median APE=90.87%, best tree

In [4]:
# A/B comparison against refined tiers without the clean average.
old_summary = pd.read_csv(TIER_ONLY_DIR / 'cv_summary_metrics.csv', dtype={'horizon_days': str})
new_summary = pd.read_csv(CLEAN_AVERAGE_DIR / 'cv_summary_metrics.csv', dtype={'horizon_days': str})
comparison_rows = []
for horizon in [str(h) for h in HORIZONS] + ['combined']:
    old = old_summary[(old_summary['horizon_days'].eq(horizon)) & (old_summary['model'].eq(MODEL_NAME))].iloc[0]
    new = new_summary[(new_summary['horizon_days'].eq(horizon)) & (new_summary['model'].eq(MODEL_NAME))].iloc[0]
    comparison_rows.append({
        'horizon_days': horizon,
        'tier_only_rmsle': old['rmsle'],
        'with_average_rmsle': new['rmsle'],
        'rmsle_improvement_pct': 100 * (old['rmsle'] - new['rmsle']) / old['rmsle'],
        'tier_only_mape_pct': old['mape_nonzero_pct'],
        'with_average_mape_pct': new['mape_nonzero_pct'],
        'mape_improvement_pct': 100 * (old['mape_nonzero_pct'] - new['mape_nonzero_pct']) / old['mape_nonzero_pct'],
        'tier_only_log_r2': old['log_r2'],
        'with_average_log_r2': new['log_r2'],
    })

comparison = pd.DataFrame(comparison_rows)
display(comparison.round(4))

importance = pd.read_csv(CLEAN_AVERAGE_DIR / 'feature_importance_gain.csv')
average_importance = importance[importance['feature'].eq(AVERAGE_FEATURE)].copy()
average_importance['gain_rank'] = average_importance.groupby('horizon_days')['gain_share_within_horizon'].rank(method='min', ascending=False)
display(average_importance[['horizon_days', 'feature', 'gain', 'gain_share_within_horizon']].round(5))

  horizon_days  tier_only_rmsle  ...  tier_only_log_r2  with_average_log_r2
0            7           2.1492  ...            0.2470               0.3070
1           14           2.1468  ...            0.2775               0.3627
2           21           2.0817  ...            0.3250               0.3305
3           30           2.0408  ...            0.3801               0.3785
4     combined           2.1096  ...            0.3033               0.3423

[5 rows x 9 columns]
     horizon_days  ... gain_share_within_horizon
1               7  ...                   0.09215
65             14  ...                   0.08535
127            21  ...                   0.01247
188            30  ...                   0.00000

[4 rows x 4 columns]


In [5]:
# Artifact and leakage tests.
manifest = json.loads((CLEAN_AVERAGE_DIR / 'training_manifest.json').read_text(encoding='utf-8'))
predictions = pd.read_csv(CLEAN_AVERAGE_DIR / 'cv_predictions.csv')
reference_predictions = pd.read_csv(TIER_ONLY_DIR / 'cv_predictions.csv')
test_rows = []

def check(name, condition, detail=''):
    test_rows.append({'test': name, 'status': 'PASS' if bool(condition) else 'FAIL', 'detail': detail})

check('reserved test remains unevaluated', manifest['status'] == 'candidate_reserved_test_not_evaluated')
check('same OOF rows as tier-only model', set(zip(predictions['horizon_days'], predictions['horizon_row_position'])) == set(zip(reference_predictions['horizon_days'], reference_predictions['horizon_row_position'])))
check('all predictions finite', np.isfinite(predictions.filter(like='predicted_').to_numpy(dtype=float)).all())

for record in manifest['models']:
    horizon = record['horizon_days']
    bundle = joblib.load(CLEAN_AVERAGE_DIR / record['model_path'])
    X, y, assignments = loaded[horizon]
    development_positions = set(assignments.loc[assignments['partition'].eq('development'), 'horizon_row_position'].astype(int))
    reserved_positions = set(assignments.loc[assignments['partition'].eq('test_reserved'), 'horizon_row_position'].astype(int))
    predicted_positions = set(predictions.loc[predictions['horizon_days'].eq(horizon), 'horizon_row_position'].astype(int))
    sample_prediction = bundle.predict_views(X.iloc[[min(development_positions)]])

    check(f'day {horizon} clean average saved in model', AVERAGE_FEATURE in bundle.feature_names)
    check(f'day {horizon} raw channel values absent', {'ch_views_at_publish', 'ch_subs_at_publish'}.isdisjoint(bundle.feature_names))
    check(f'day {horizon} development coverage exact', predicted_positions == development_positions)
    check(f'day {horizon} reserved rows absent', predicted_positions.isdisjoint(reserved_positions))
    check(f'day {horizon} bundle reloads and predicts', len(sample_prediction) == 1 and np.isfinite(sample_prediction).all())

tests = pd.DataFrame(test_rows)
display(tests)
failures = tests[tests['status'].eq('FAIL')]
assert failures.empty, failures.to_string(index=False)
print(f'PASS: all {len(tests)} clean-average model checks succeeded.')

                                   test status detail
0     reserved test remains unevaluated   PASS       
1      same OOF rows as tier-only model   PASS       
2                all predictions finite   PASS       
3    day 7 clean average saved in model   PASS       
4       day 7 raw channel values absent   PASS       
5      day 7 development coverage exact   PASS       
6            day 7 reserved rows absent   PASS       
7     day 7 bundle reloads and predicts   PASS       
8   day 14 clean average saved in model   PASS       
9      day 14 raw channel values absent   PASS       
10    day 14 development coverage exact   PASS       
11          day 14 reserved rows absent   PASS       
12   day 14 bundle reloads and predicts   PASS       
13  day 21 clean average saved in model   PASS       
14     day 21 raw channel values absent   PASS       
15    day 21 development coverage exact   PASS       
16          day 21 reserved rows absent   PASS       
17   day 21 bundle reloads a

## Checkpoint decision

Retain the clean average only if the grouped cross-validation comparison improves. Day 30 has no valid pre-publication values, so its importance must be zero and its result should remain effectively unchanged apart from normal training variation.